# Nemotron LoRA — train on Kaggle (free 2×T4, 4-bit QLoRA)

**Reality check:** the T4 (sm_75) can't run the fused Mamba kernel, so it uses the
pure-PyTorch `torch_forward` path — correct, but **slow (~20–60 s/step)**. One 12 h
session trains only a *fraction* of an epoch. The adapter is saved to
`/kaggle/working/lora_adapter` every few steps, and you can **resume across sessions**
(see the last section) to keep improving it for free.


## 0. Sanity-check the GPUs (expect **2× Tesla T4**)

If you don't see two T4s here, fix **Settings → Accelerator → GPU T4 ×2** before the
long run — otherwise the 17 GB model won't fit on a single 16 GB card.


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv
import torch
n = torch.cuda.device_count()
caps = [torch.cuda.get_device_capability(i) for i in range(n)]
print(f'torch sees {n} GPU(s); compute caps = {caps}')
assert n >= 2, 'Need 2x T4 — set Accelerator to "GPU T4 x2".'
if caps and caps[0] >= (8, 0):
    print('Ampere+ detected (fast Mamba kernel available) — you could use the A100/Colab path.')
else:
    print('sm_75 (T4): torch_forward will be auto-forced — correct but slow (~20-60 s/step).')


## 1b. OFFLINE mode (Internet OFF) — run this, then SKIP sections 1 & 2

Auto-detects three attached datasets and sets everything up with **no network**:
`nemotron-pipeline-repo` (code), `nemotron-libs` (wheels), and the Nemotron **model**
(attach via *+ Add Data -> Models*). If running with **Internet ON**, skip this cell
and use sections 1 & 2 instead.


In [ ]:
import os, glob, shutil, importlib
def find(*pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h: return h[0]
    return None

# 1) repo -> copy to a writable dir
hit = find('/kaggle/input/**/scripts/03_train_lora.py')
assert hit, 'Attach the nemotron-pipeline-repo dataset.'
repo_src = hit.split('/scripts/')[0]
os.chdir('/kaggle/working'); shutil.rmtree('repo', ignore_errors=True)
shutil.copytree(repo_src, 'repo'); os.chdir('/kaggle/working/repo')
print('repo  ->', repo_src)

# 2) libs -> offline wheel install (pip auto-picks cp311/cp312)
W = find('/kaggle/input/*nemotron-libs*/mamba_ssm-*.whl', '/kaggle/input/**/mamba_ssm-*.whl')
assert W, 'Attach the nemotron-libs dataset.'
W = os.path.dirname(W)
os.system(f'pip install -q --no-index --no-deps {W}/trl-*.whl {W}/peft-*.whl {W}/einops-*.whl')
os.system(f'pip install -q --no-index --no-deps {W}/mamba_ssm-*.whl {W}/causal_conv1d-*.whl')
for m in ('mamba_ssm','causal_conv1d','trl','peft'): importlib.import_module(m)
print('libs  ->', W, '(offline install OK)')

# 3) model -> set MODEL_PATH from a mounted folder with config.json + *.safetensors
MODEL = None
for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
    d = os.path.dirname(cfg)
    if glob.glob(d + '/*.safetensors'): MODEL = d; break
if MODEL:
    os.environ['MODEL_PATH'] = MODEL
    os.environ['HF_HUB_OFFLINE'] = '1'; os.environ['TRANSFORMERS_OFFLINE'] = '1'
    print('model ->', MODEL, '(offline)')
else:
    print('NO model dataset mounted -> attach the Nemotron model, or run with Internet ON.')


## 1. Code + dependencies (keep Kaggle's torch 2.10)

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil einops
# we never use vision; a mismatched torchvision breaks transformers' import -> remove it
!pip uninstall -y -q torchvision torchaudio
import torch
print("TORCH:", torch.__version__, "abi:", torch.compiled_with_cxx11_abi())

## 2. Install mamba_ssm only (torch_forward path; skips the T4-incompatible SSD kernel)

In [ ]:
# Install ONLY mamba_ssm (needed at import for rmsnorm), NOT causal_conv1d.
# Without causal_conv1d the model uses its torch_forward path, which skips the
# Mamba-2 SSD Triton kernel that fails to compile on T4 (sm_75).
!pip install -q --no-deps 'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'
!python -c "import mamba_ssm; print('mamba_ssm OK; causal_conv1d intentionally absent -> torch_forward')"

## 3. Competition data (recursive find)

In [ ]:
import os, glob, shutil, urllib.request
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
if hits:
    shutil.copy(hits[0], 'data/train.csv'); print('train.csv (mount) <-', hits[0])
else:
    TOK = "KGAT_xxxxxxxxxxxxxxxx"   # <-- your Kaggle API token (Colab/local path)
    url='https://www.kaggle.com/api/v1/competitions/data/download/nvidia-nemotron-model-reasoning-challenge/train.csv'
    req=urllib.request.Request(url, headers={'Authorization': f'Bearer {TOK}'})
    open('data/train.csv','wb').write(urllib.request.urlopen(req).read())
    print('train.csv (download):', os.path.getsize('data/train.csv'), 'bytes')

## 4. EDA + build the SFT data

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 5. Train — auto-detects your GPU and picks the config

- **Big Ampere+/Ada/Blackwell (≥40 GB, e.g. RTX PRO 6000 96 GB):** full **bf16, no
  quant**, `adamw_torch` — fastest, no bitsandbytes, full epoch ≈ 2 h.
- **Smaller sm_80+ GPU:** 4-bit QLoRA.
- **T4 (sm_75):** 4-bit + checkpointing + `torch_forward` (slow ~98 s/step).

On brand-new **Blackwell (sm_120)**, if the fused Mamba kernel errors, uncomment the
`FORCE_TORCH_FORWARD` line and re-run — still fast on that card.


In [ ]:
import os, torch
p = torch.cuda.get_device_properties(0)
vram = p.total_memory/1e9
os.environ['SAVE_STEPS'] = '100'
os.environ['NUM_EPOCHS'] = '1'

if p.major >= 8 and vram >= 40:           # A100-40/80, RTX 6000 Ada/Blackwell, etc.
    os.environ['QUANT'] = 'none'           # full bf16 fits — no quantization at all
    os.environ['OPTIM'] = 'adamw_torch'    # no bitsandbytes dependency
    os.environ['NEMOTRON_MAX_MEMORY_GPU'] = f'{int(vram)-6}GiB'
    os.environ['SFT_MAX_SEQ_LENGTH'] = '1024'
    print(f'bf16 no-quant on {p.name} ({vram:.0f} GB, sm_{p.major}{p.minor})')
elif p.major >= 8:                          # smaller Ampere+ -> 4-bit QLoRA
    os.environ['QUANT'] = '4bit'
    os.environ['SFT_MAX_SEQ_LENGTH'] = '768'
    print(f'4-bit QLoRA on {p.name} ({vram:.0f} GB)')
else:                                        # T4 (sm_75) -> slow torch_forward path
    os.environ['QUANT'] = '4bit'
    os.environ['GRAD_CHECKPOINT'] = '1'
    os.environ['GRAD_ACCUM'] = '4'
    os.environ['SFT_MAX_SEQ_LENGTH'] = '512'
    os.environ['SAVE_STEPS'] = '25'
    os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '13GiB'
    print(f'4-bit + torch_forward on {p.name} (slow ~98 s/step)')

# Blackwell sm_120: uncomment if the fused Mamba kernel errors ->
# os.environ['FORCE_TORCH_FORWARD'] = '1'

!python scripts/03_train_lora.py --data-path data/train_sft.jsonl \
    --output-dir /kaggle/working/lora_adapter --no-smoke


## 6. Package the submission — and how to RESUME next session

`submission.zip` (rank 16 ≤ 32 ✓) is built from the latest adapter; download it from
the **Output** tab and submit.

**To train more for free next session:** after this run, click *Save Version →
Save & Run All*; the `/kaggle/working/lora_adapter` folder becomes part of the
notebook output. Create a **Dataset** from that output (or *+ Add Data → your
notebook output*), attach it to a new session, and re-run — the train cell auto-detects
it via `RESUME_ADAPTER` and continues from where you stopped.


In [ ]:
!python scripts/05_package_submission.py --adapter-dir /kaggle/working/lora_adapter --output /kaggle/working/submission.zip